# QLoRA fine-tune Qwen2.5-7B trên synthetic data (Colab)
Fine-tune LLM extractor bằng `data/synthetic/train_sft.jsonl` (self-host ≤9B, KHÔNG API ngoài).
Runtime → GPU (T4 chạy được 4-bit; **L4/A100 nhanh hơn nhiều**). Lưu adapter vào Drive vì Colab hay ngắt.

**Quy trình:** cài env → smoke-train (ít bước) kiểm OOM → train full → trỏ config → đo dev so với few-shot.

In [ ]:
# 1) Drive: cache model + LƯU ADAPTER bền vững (Colab ngắt là mất /content)
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
ADAPTER = '/content/drive/MyDrive/models/qwen7b-lora'   # adapter lưu ở Drive
print('HF_HOME =', os.environ['HF_HOME'], '| ADAPTER =', ADAPTER)

In [ ]:
# 2) Lấy code
%cd /content
![ -d viettel-medreason ] || git clone https://github.com/tienndat1904/viettel-medreason.git
%cd /content/viettel-medreason
!git pull -q

In [ ]:
# 3) Cài env — KHÔNG cài lại transformers; core + update bitsandbytes/peft/accelerate
!pip install -q -r requirements.txt
!pip install -q -U bitsandbytes peft accelerate
import torch, transformers, bitsandbytes as bnb, peft
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| bnb', bnb.__version__, '| peft', peft.__version__,
      '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
# 4) (Tùy chọn) sinh lại synthetic data — đã ship sẵn trong repo, chỉ chạy nếu muốn tái tạo
# !python src/datagen/gen_synthetic.py --n 1500 --seed 42
!wc -l data/synthetic/train_sft.jsonl

In [ ]:
# 5) SMOKE-TRAIN: chạy vài chục bước để chắc KHÔNG OOM + đường ống train chạy được
#    (epochs nhỏ; nếu T4 OOM -> giảm --max-len 1536, hoặc dùng --model Qwen/Qwen2.5-3B-Instruct)
!python scripts/train_qlora.py --data data/synthetic/train_sft.jsonl \
  --out /content/smoke-lora --epochs 0.05 --max-len 2048 --seed 42

In [ ]:
# 6) TRAIN FULL -> lưu adapter thẳng vào Drive (ADAPTER). ~vài giờ trên T4; nhanh hơn trên L4/A100.
!python scripts/train_qlora.py --data data/synthetic/train_sft.jsonl \
  --out "$ADAPTER" --epochs 2 --batch 1 --grad-accum 16 --lr 1e-4 --max-len 2048 --seed 42
!ls -la "$ADAPTER"

In [ ]:
# 7) Trỏ config sang adapter + backend llm (giữ nguyên comment không cần thiết)
import yaml
cfg = yaml.safe_load(open('configs/config.yaml', encoding='utf-8'))
cfg['extract']['backend'] = 'llm'
cfg['extract']['lora_adapter'] = ADAPTER
yaml.safe_dump(cfg, open('configs/config.yaml', 'w', encoding='utf-8'),
               allow_unicode=True, sort_keys=False)
print('extract.lora_adapter =', cfg['extract']['lora_adapter'])

In [ ]:
# 8) Chạy pipeline (QLoRA) trên 30 file dev để so với few-shot, rồi đo
import os, shutil, glob
os.makedirs('devset/input', exist_ok=True)
for g in glob.glob('data/dev/gold/*.json'):
    n = os.path.basename(g)[:-5]
    shutil.copy(f'data/test/input/{n}.txt', f'devset/input/{n}.txt')
!python src/pipeline.py --input devset/input --output dev_pred --backend llm
!python src/eval/scorer.py --pred dev_pred --gold data/dev/gold --mode overlap
!python src/eval/scorer.py --pred dev_pred --gold data/dev/gold --mode exact

In [ ]:
# 9) (Khi đã hài lòng) chạy full 100 file + đóng gói + tải về
!python src/pipeline.py --input data/test/input --output output --backend llm
!python scripts/package_submission.py --output output --input data/test/input --n 100
from google.colab import files; files.download('output.zip')